In [1]:
# @title Environment setup
%set_env MLFLOW_TRACKING_URI https://dagshub.com/m09/landscape-classifier.mlflow
%set_env MPLBACKEND notebook
!rm -rf sample_data .config
!pip install -U uv
!git config --global user.email "jane@doe.eu"
!git config --global user.name "Jane Doe"
!git config --global init.defaultBranch main
!git clone --branch solution-en https://github.com/shuuchuu/landscape-classifier.git
%cd /content/landscape-classifier
!uv pip install --system -e ".[fastapi]"

env: MLFLOW_TRACKING_URI=https://dagshub.com/m09/landscape-classifier.mlflow
env: MPLBACKEND=notebook
Cloning into 'landscape-classifier'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 71 (delta 29), reused 57 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (71/71), 4.72 MiB | 9.63 MiB/s, done.
Resolving deltas: 100% (29/29), done.
/content/landscape-classifier
Using Python 3.13.15 environment at: /usr
Resolved 201 packages in 60ms
   Building landscape-classifier @ file:///content/landscape-classifier         ⠙ Preparing packages... (0/1)                                                   warning: Found license classifier `License :: OSI Approved :: Apache Software License`. License classifiers are ambiguous and deprecated per PEP 639; projects should use `project.license` and `project.license-files` instead.
Prepared 1 package in 4ms
Uninstalled 1 package in 0.41ms
Instal

# MLflow, model registry & deployment

This demonstration walks through the solution of the lab of the same name, on the `solution-en` branch of the [`landscape-classifier` repository](https://github.com/shuuchuu/landscape-classifier/tree/solution-en): publishing a model to the MLflow model registry, using this registry, then serving the model with a FastAPI API packaged with Docker.

## MLflow credentials

Enter your DagsHub credentials (the password is the secret token, not your DagsHub password):

In [2]:
username = ""
token = ""

In [3]:
# @title To run
%set_env MLFLOW_TRACKING_USERNAME {username}
%set_env MLFLOW_TRACKING_PASSWORD {token}

## Data retrieval

In [4]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s landscape
!mv landscape/seg_train train-data

remote: Enumerating objects: 2, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 2 (delta 0), reused 1 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (2/2), 1.94 KiB | 1.94 MiB/s, done.
remote: Enumerating objects: 24289, done.
remote: Counting objects: 100% (24289/24289), done.
remote: Compressing objects: 100% (24289/24289), done.
remote: Total 24289 (delta 0), reused 24289 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (24289/24289), 341.65 MiB | 18.67 MiB/s, done.
Updating files: 100% (24336/24336), done.
✓ landscape — 24336 fichiers, 401M


## Publishing a model to a model registry

The training script tracks metrics with [MLflow Tracking](https://mlflow.org/docs/latest/tracking.html) (`autolog`), as well as the training dataset. It then wraps the Keras model in a custom `pyfunc` model (`WrappedModel`) that embeds the image preprocessing, publishes it to the model registry as `dev.ml.landscape-classifier`, and gives the new version the `champion` alias.

In [5]:
!cat src/landscape_classifier/train.py

import os
from io import BytesIO
from pathlib import Path
from subprocess import run
from typing import Any

import yaml
from keras import Model, Sequential, layers, optimizers
from mlflow import MlflowClient, log_input, set_experiment, start_run
from mlflow.data.numpy_dataset import from_numpy
from mlflow.keras import autolog
from mlflow.pyfunc import PythonModel, PythonModelContext, log_model
from numpy import array, vstack
from pydantic import BaseModel

from landscape_classifier.data import LABEL_NAMES, get_images, process_image


def get_lenet(image_size: tuple[int, int], learning_rate: float) -> Model:
    def conv(filters: int, padding: str) -> layers.Conv2D:
        return layers.Conv2D(
            filters=filters, kernel_size=5, padding=padding, activation="sigmoid"
        )

    def pooling() -> layers.MaxPooling2D:
        return layers.MaxPooling2D()

    def dense(units: int, activation: str = "sigmoid") -> layers.Dense:
        return layers.Dense(units, activation=acti

Its parameters are managed by DVC, in `params.yaml`:

In [6]:
!cat params.yaml

train_dir: train-data
image_size: [100, 100]
learning_rate: 0.0001
experiment: Landscape Classification
name: model
model_name: dev.ml.landscape-classifier
model_alias: champion
epochs: 3


In [7]:
!landscape-classifier

2026-10-08 20:08:35.230060: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026/10/08 20:08:42 INFO mlflow.tracking.fluent: Experiment with name 'Landscape Classification' does not exist. Creating a new experiment.
Traitement des dossiers:   0% 0/6 [00:00<?, ?it/s]
Dossier buildings:   0% 0/2191 [00:00<?, ?it/s]
Dossier buildings:   5% 104/2191 [00:00<00:02, 1032.55it/s]
Dossier buildings:  10% 212/2191 [00:00<00:01, 1059.43it/s]
Dossier buildings:  15% 320/2191 [00:00<00:01, 1068.12it/s]
Dossier buildings:  20% 429/2191 [00:00<00:01, 1074.14it/s]
Dossier buildings:  25% 540/2191 [00:00<00:01, 1084.52it/s]
Dossier buildings:  30% 649/2191 [00:00<00:01, 1076.09it/s]
Dossier buildings:  35% 759/2191 [00:00<00:01, 1082.48it/s]
Dossier buildings:  40% 869/2

## Using the MLflow model registry

### Creating an MLflow client

The tracking address is correctly read from the environment variable defined at the top of this notebook:

In [8]:
import mlflow

MODEL_NAME = "dev.ml.landscape-classifier"

client = mlflow.MlflowClient()
client.tracking_uri

'https://dagshub.com/m09/landscape-classifier.mlflow'

### Model search

All the registered models, each with its most recent version:

In [9]:
def latest_version(name: str) -> str:
  versions = client.search_model_versions(f"name = '{name}'")
  return str(max(int(v.version) for v in versions))


for registered_model in client.search_registered_models():
  print(registered_model.name, latest_version(registered_model.name))

dev.ml.landscape-classifier 1


### Retrieving a model from the model registry

An MLflow model can have several *flavors*, as many ways to load it. Ours was published as a custom `pyfunc` model: its only flavor is `python_function`, which gives the whole pipeline (image preprocessing, prediction, result formatting). A Keras flavor would only have given the network, without the preprocessing.

In [10]:
from pathlib import Path

version = latest_version(MODEL_NAME)
model = mlflow.pyfunc.load_model(model_uri=f"models:/{MODEL_NAME}/{version}")
print(list(model.metadata.flavors))
model.predict([Path("forest1.jpg").read_bytes(), Path("forest2.jpg").read_bytes()])

2026/10/08 20:10:27 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - absl-py (current: 1.4.0, required: absl-py==2.5.0)
 - aiohttp (current: 3.14.3, required: aiohttp==3.14.4)
 - anyio (current: 4.14.2, required: anyio==4.15.1)
 - atpublic (current: 5.1, required: atpublic==8.0.1)
 - cachetools (current: 6.2.6, required: cachetools==7.2.1)
 - charset-normalizer (current: 3.4.9, required: charset-normalizer==3.5.2)
 - contourpy (current: 1.3.3, required: contourpy==1.4.0)
 - cryptography (current: 50.0.1, required: cryptography==50.0.2)
 - fastapi (current: 0.141.1, required: fastapi==0.143.0)
 - filelock (current: 3.32.5, required: filelock==4.0.12)
 - fonttools (current: 4.64.0, required: fonttools==4.66.1)
 - fsspec (current: 2025.12.0, required: fsspec==2026.9.0)
 - google-auth (current: 2.49.0, required: google-auth==2.61.0)
 - grpcio (current: 1.83.1, required: grpcio==1.84.0)
 - h5py (

['python_function']
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step


ClassificationResult(predicted=['street', 'buildings'], probabilities=[{'buildings': 0.10004390776157379, 'forest': 0.35625600814819336, 'glacier': 0.024229787290096283, 'mountain': 0.017927944660186768, 'sea': 0.03804943338036537, 'street': 0.4634930193424225}, {'buildings': 0.3924207091331482, 'forest': 0.05892636626958847, 'glacier': 0.07465572655200958, 'mountain': 0.2037796527147293, 'sea': 0.13926197588443756, 'street': 0.13095563650131226}])

### Adding an alias & tags

Aliases and tags are a way to organize models in an MLflow model registry. Aliases always target a specific version of a model while tags can also apply to all versions of a model.

The training script already gave the `champion` alias to the last version. We add a `domain` tag with value `cv` to the model and a `size` tag with value `small` to its last version:

In [11]:
client.set_registered_model_tag(MODEL_NAME, "domain", "cv")
client.set_model_version_tag(MODEL_NAME, version, "size", "small")
client.get_model_version_by_alias(MODEL_NAME, "champion")

<ModelVersion: aliases=['champion'], creation_timestamp=1791490210831, current_stage='None', deployment_job_state=None, description='LeNet entraîné sur `train-data` pendant 3 époque(s), val_accuracy=0.5357.', last_updated_timestamp=1791490212305, metrics=None, model_id=None, name='dev.ml.landscape-classifier', params=None, run_id='36e98a865272450b9d333f6b69c10caf', run_link='', source='models:/m-02428ac174a747aeb528c5b9292e9e2f', status='READY', status_message=None, tags={'git.commit': 'c862d552aac75b500c3fb683c59d115b96b1dcbb', 'size': 'small'}, user_id='', version='1', workspace='default'>

We can then retrieve the model by its alias rather than its exact version:

In [12]:
model = mlflow.pyfunc.load_model(model_uri=f"models:/{MODEL_NAME}@champion")
model.predict([Path("forest1.jpg").read_bytes()])

2026/10/08 20:10:40 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - absl-py (current: 1.4.0, required: absl-py==2.5.0)
 - aiohttp (current: 3.14.3, required: aiohttp==3.14.4)
 - anyio (current: 4.14.2, required: anyio==4.15.1)
 - atpublic (current: 5.1, required: atpublic==8.0.1)
 - cachetools (current: 6.2.6, required: cachetools==7.2.1)
 - charset-normalizer (current: 3.4.9, required: charset-normalizer==3.5.2)
 - contourpy (current: 1.3.3, required: contourpy==1.4.0)
 - cryptography (current: 50.0.1, required: cryptography==50.0.2)
 - fastapi (current: 0.141.1, required: fastapi==0.143.0)
 - filelock (current: 3.32.5, required: filelock==4.0.12)
 - fonttools (current: 4.64.0, required: fonttools==4.66.1)
 - fsspec (current: 2025.12.0, required: fsspec==2026.9.0)
 - google-auth (current: 2.49.0, required: google-auth==2.61.0)
 - grpcio (current: 1.83.1, required: grpcio==1.84.0)
 - h5py (

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 654ms/step


ClassificationResult(predicted=['street'], probabilities=[{'buildings': 0.10003605484962463, 'forest': 0.35624128580093384, 'glacier': 0.024225102737545967, 'mountain': 0.017935052514076233, 'sea': 0.038052402436733246, 'street': 0.4635101854801178}])

### Changing model phases

To promote a model in the production pipeline, the recommended way is to copy the chosen version to a new model name, here the one of the staging environment:

In [13]:
client.copy_model_version(
  src_model_uri=f"models:/{MODEL_NAME}@champion",
  dst_name="staging.ml.landscape-classifier",
)

Successfully registered model 'staging.ml.landscape-classifier'.
Copied version '1' of model 'dev.ml.landscape-classifier' to version '1' of model 'staging.ml.landscape-classifier'.


<ModelVersion: aliases=[], creation_timestamp=1791490242602, current_stage='None', deployment_job_state=None, description='LeNet entraîné sur `train-data` pendant 3 époque(s), val_accuracy=0.5357.', last_updated_timestamp=1791490242602, metrics=None, model_id=None, name='staging.ml.landscape-classifier', params=None, run_id='36e98a865272450b9d333f6b69c10caf', run_link='', source='models:/124456_dev.ml.landscape-classifier/1', status='READY', status_message=None, tags={'git.commit': 'c862d552aac75b500c3fb683c59d115b96b1dcbb', 'size': 'small'}, user_id='', version='1', workspace='default'>

## API creation with FastAPI

The preprocessing is the `process_image` function, used both for training and for inference (in `WrappedModel`): this is the point that needs extreme care, since a different preprocessing between training and inference degrades predictions without raising any error.

In [14]:
!cat src/landscape_classifier/data.py

import pathlib
import typing

import numpy
import tqdm
from PIL import Image
from sklearn.model_selection import train_test_split

LABEL_NAMES = ["buildings", "forest", "glacier", "mountain", "sea", "street"]
LABEL_TO_INDEX = {label: i for i, label in enumerate(LABEL_NAMES)}


def process_image(
    file: typing.BinaryIO | str | pathlib.Path, image_size: tuple[int, int]
) -> numpy.ndarray:
    return numpy.array(Image.open(file).resize(image_size))[None, ...]


def get_images(
    dir_path: pathlib.Path, image_size: tuple[int, int]
) -> tuple[numpy.ndarray, numpy.ndarray, numpy.ndarray, numpy.ndarray]:
    images = []
    labels = []

    for subdir_path in tqdm.tqdm(
        list(dir_path.iterdir()), desc="Traitement des dossiers"
    ):
        dir_name = subdir_path.name

        label = LABEL_TO_INDEX.get(dir_name)

        for image_path in tqdm.tqdm(
            list(subdir_path.iterdir()), desc=f"Dossier {dir_name}", leave=False
        ):
            images.append(process_image

The API loads the model only once, at startup, from the registry (by its alias), describes its return type with [Pydantic](https://docs.pydantic.dev/latest/) and accepts images [sent with POST](https://fastapi.tiangolo.com/tutorial/request-files/):

In [15]:
!cat src/landscape_classifier/api.py

from collections.abc import AsyncIterator
from contextlib import asynccontextmanager
from typing import Final, Literal, TypedDict, cast

from fastapi import FastAPI, HTTPException, UploadFile
from mlflow.exceptions import MlflowException
from mlflow.pyfunc import PyFuncModel, load_model
from pydantic import BaseModel

from landscape_classifier.data import LABEL_NAMES

MODEL_NAME: Final = "dev.ml.landscape-classifier"
MODEL_ALIAS: Final = "champion"

model: PyFuncModel | None = None


@asynccontextmanager
async def lifespan(_: FastAPI) -> AsyncIterator[None]:  # noqa: RUF029
    global model
    try:
        model = load_model(model_uri=f"models:/{MODEL_NAME}@{MODEL_ALIAS}")
    except MlflowException:
        model = None
    yield


app = FastAPI(lifespan=lifespan)


Probabilities = TypedDict("Probabilities", dict.fromkeys(LABEL_NAMES, float))  # type: ignore


class ClassificationResult(BaseModel):
    predicted: list[Literal[tuple(LABEL_NAMES)]]  # type: ignore
    probabilities: li

### Starting the API

We start the server in the background, then wait for it to answer (loading the model can take a while):

In [16]:
!nohup uvicorn landscape_classifier.api:app --port 8000 > api.log 2>&1 &

In [17]:
import time

import requests

for _ in range(300):
  try:
    requests.get("http://127.0.0.1:8000/docs", timeout=1)
    break
  except requests.ConnectionError:
    time.sleep(1)
print(Path("api.log").read_text())

INFO:     Started server process [23579]
INFO:     Waiting for application startup.

2026/10/08 20:10:54 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - absl-py (current: 1.4.0, required: absl-py==2.5.0)
 - aiohttp (current: 3.14.3, required: aiohttp==3.14.4)
 - anyio (current: 4.14.2, required: anyio==4.15.1)
 - atpublic (current: 5.1, required: atpublic==8.0.1)
 - cachetools (current: 6.2.6, required: cachetools==7.2.1)
 - charset-normalizer (current: 3.4.9, required: charset-normalizer==3.5.2)
 - contourpy (current: 1.3.3, required: contourpy==1.4.0)
 - cryptography (current: 50.0.1, required: cryptography==50.0.2)
 - fastapi (current: 0.141.1, required: fastapi==0.143.0)
 - filelock (current: 3.32.5, required: filelock==4.0.12)
 - fonttools (current: 4.64.0, required: fonttools==4.66.1)
 - fsspec (current: 2025.12.0, required: fsspec==2026.9.0)
 - google-auth (current: 2.49.0, required:

### Calling the API

In [18]:
with open("forest1.jpg", "rb") as forest1, open("forest2.jpg", "rb") as forest2:
  response = requests.post(
    "http://127.0.0.1:8000/",
    files=[("images", forest1), ("images", forest2)],
  )
response.raise_for_status()
response.json()

{'predicted': ['street', 'buildings'],
 'probabilities': [{'buildings': 0.10004390776157379,
   'forest': 0.35625600814819336,
   'glacier': 0.024229787290096283,
   'mountain': 0.017927944660186768,
   'sea': 0.03804943338036537,
   'street': 0.4634930193424225},
  {'buildings': 0.3924207091331482,
   'forest': 0.05892636626958847,
   'glacier': 0.07465572655200958,
   'mountain': 0.2037796527147293,
   'sea': 0.13926197588443756,
   'street': 0.13095563650131226}]}

## Dockerfile creation

Docker isn't available in Colab: here is the Dockerfile of the solution. A first stage exports the dependencies with `uv`, the second one installs them on the Python 3.12 base image along with the project, then starts the FastAPI server.

In [19]:
!cat Dockerfile

FROM python:3.12-slim-trixie AS base
WORKDIR /app

FROM base AS uv
COPY --from=ghcr.io/astral-sh/uv:0.12.14 /uv /uvx /bin/
COPY uv.lock pyproject.toml /app/
RUN uv export --extra fastapi --frozen --no-dev --no-emit-project -o requirements.txt

FROM base AS runtime
COPY --from=uv /app/requirements.txt /app/requirements.txt
RUN pip install --no-cache-dir -r requirements.txt
COPY ./pyproject.toml ./README.md /app/
COPY ./params.yaml /app/
COPY ./src /app/src/
RUN pip install .
CMD ["uvicorn", "landscape_classifier.api:app", "--host", "0.0.0.0", "--port", "80"]


On a machine with Docker, we build then run the image, passing it the MLflow credentials as environment variables:

    docker build -t shuuchuu/landscape-classifier .
    docker run --rm -it -e MLFLOW_TRACKING_URI -e MLFLOW_TRACKING_USERNAME \
      -e MLFLOW_TRACKING_PASSWORD -p 8000:80 shuuchuu/landscape-classifier

The API documentation, which also serves as a test client, is then at <http://localhost:8000/docs>.

## API improvements

Several choices of the solution make the API suited to production deployment with a Docker image:

- the model is loaded only once, at startup, not on every request;
- the MLflow credentials are given when starting the container, never copied into the image;
- the API answers 503 if the model couldn't be loaded.

We could go further by making the model name and alias configurable with environment variables, to serve the same image in development and in staging.